# Lesson 3.3: Plotly Practice

**🎬 Video:** [Lesson 3.3: Plotly Practice](#)

## Overview

In this practice lesson, you will experiment with the visualizations introduced in [Lesson 3.2](lesson_3_2_plotly_visualization.ipynb) and build new ones from scratch. You will:

- Load the pre-cleaned pickle file
- Modify and experiment with existing visualizations
- Explore Plotly styling: markers, color swatches, templates, and text case
- Create a new visualization showing keyword trends over time

**Prerequisites:** Complete [Lesson 3.2 — Visualizing Data with Plotly](lesson_3_2_plotly_visualization.ipynb) first.

## 📖 1 Follow Along — Loading Libraries and Data

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

First, load the libraries and the pre-cleaned pickle file. Then you will recreate the first visualization from Lesson 3.2 and examine how changing the time interval changes the chart.

In [ ]:
# Import required libraries
import pandas as pd
import plotly.express as px

# Load the pre-cleaned data
df = pd.read_pickle('../data/JMU/JMU_raw.pickle')
df.head()

The `dt.to_period()` method converts a datetime column into a **period** — a label that represents a chunk of time. The chunk size is controlled by the string you pass in:

| Code | Period | Example output |
|------|--------|----------------|
| `dt.to_period('D')` | Day | `2024-03-18` |
| `dt.to_period('M')` | Month | `2024-03` |
| `dt.to_period('Y')` | Year | `2024` |

Switching from monthly to yearly is as simple as changing `'M'` to `'Y'` — the rest of the code stays the same.

### Why name the column `interval`?

In Lesson 3 the column was called `year_month`, which made sense when we were always grouping by month. But if you change `'M'` to `'D'` or `'Y'`, the name `year_month` is suddenly misleading.

By naming the column `interval` instead, the name stays accurate no matter what period you choose. You only need to change one value — the period string — and the chart updates automatically without any confusing mismatches between the variable name and what it actually contains.

> 💡 **Reflection:** What do we expect to happen to the count line as we change the measurement interval? What will make the line more jagged: days or years? What will make the line smoother: days or years?

In [4]:

df['interval'] = df['date'].dt.to_period('M')

# Group the data by year-month and post type, then count entries in each group
interval_counts = df.groupby(['interval', 'type'], observed=True).size().reset_index(name='count')

# Convert year_month back to datetime format so Plotly can understand it
interval_counts['interval'] = interval_counts['interval'].dt.to_timestamp()

# Create the original line chart
fig = px.line(interval_counts, 
              x='interval', 
              y='count', 
              color='type',
              title='Original: Posts and Comments per Month on r/JMU',
              labels={'count': 'Number of Posts/Comments', 'year_month': 'Year-Month'},
              markers=True)

fig.show()



### ✍️ 1.1 Critical Activity - Modifying a Plotly Chart

Plotly is a powerful visualization library that allows you to make interactive graphics with relatively few lines of code. The basic format is always the same. You create a Plotly object by running a function:

```python

figure = px.chart_type(dataframe,
                        x='column_representing_x_data',
                        y='column_representing_y_data', 
                        color='column_indicating_different_data_groups')

```
> 👉 **Note:** *If you do not enter styling information and labels, Plotly leaves these blank. All key=value pairs (i.e. `color='type'`) should be followed by a comma except for the last one.*

In [5]:
# YOUR TURN: Modify the visualization here
# Try changing px.line to px.bar or px.area
# Experiment with labels and titles

# Example modification (you can change this):
fig_modified = px.line(interval_counts, 
                      x='interval', 
                      y='count', 
                      color='type',
                      title='YOUR TITLE HERE',
                      labels={'count': 'Your Y-Label', 'interval': 'Your X-Label'})

fig_modified.show()

### Customizing your chart: Markers

Like most major libraries, Plotly has extensive documentation. 
You can use this documentation to figure out most things. Let's see if we can add markers to our chart. The example is here: [add markers](https://plotly.com/python/line-charts/#line-charts-with-markers)


In [6]:
fig_modified = px.line(
    interval_counts,
    x="interval",
    y="count",
    color="type",
    title="YOUR TITLE HERE",
    labels={"count": "Your Y-Label", "interval": "Your X-Label"},
    #modify code to add markers (see link above)
)

fig_modified.show()

### Customizing your chart: Colors

A colorway is a standard sequence of colors. By default, plotly uses it's own, but there are many more built in.

You can see all the color swatches with the commands:

```python
fig = px.colors.qualitative.swatches()
fig.show()
```
Run the code block below and see what happens.

In [7]:
fig = px.colors.qualitative.swatches()
fig.show()

> 👉 **Note:** *These are all qualitative color swatches. This is good for variables that are categories.*

We can use these swatches in our charts by setting the value:  
`color_discrete_sequence=px.colors.qualitative.G10`

The value after the last dot is the swatch. In this case `G10`.

Modify the chart below to add your own color swatch.

In [8]:
fig_modified = px.line(
    interval_counts,
    x="interval",
    y="count",
    color="type",
    title="YOUR TITLE HERE",
    labels={"count": "Your Y-Label", "interval": "Your X-Label"},
    #Add color swatch here 
)

fig_modified.show()

---

## 2 Review and Modify Visualization 2 - Keyword Scores

Let's recreate the keyword analysis and tweak the layout to simplify reading the chart.

In [9]:
# Recreate the keyword analysis from Lesson 3
keywords = ['tuition', 'covid', 'party', 'football', 'class', 'library', 'campus']

# Helper function to check if text contains a keyword
def contains_keyword(text, keyword):
    if pd.isna(text):
        return False
    return keyword.lower() in text.lower()

# Calculate average scores for each keyword
keyword_scores = []
for keyword in keywords:
    has_keyword_mask = df['text'].apply(lambda x: contains_keyword(x, keyword))
    avg_score = df[has_keyword_mask]['score'].mean()
    keyword_scores.append({'keyword': keyword, 'score': avg_score})

keyword_df = pd.DataFrame(keyword_scores)

# Original bar chart - unsorted
fig = px.bar(keyword_df, 
             x='keyword', 
             y='score',
             title='Original: Average Post Scores by Keyword')
fig.show()

> 💡 **Reflection:** What are some issues with the layout of this chart?

### ✍️ 2.1 Critical Activity - Sorting Keyword Scores

The chart above shows keywords in their original order, but it's hard to compare scores. Let's improve this visualization using Plotly's `update_layout()` method.

`update_layout()` allows you to modify the layout and styling of a figure after it has been created. As per usual, there are a ton of options, but we'll focus on a common annoyance: category order and category names. 

#### Category Order

In [10]:
# Step 1: Create the chart
fig_sorted = px.bar(keyword_df, 
                    x='keyword', 
                    y='score',
                    title='Plotly Sorting: Keywords by Score Value',
                    labels={'keyword': 'Keywords', 'score': 'Average Score'})

# Step 2: Sort the x-axis by the y-values (scores) 
fig_sorted.update_layout(
    xaxis={'categoryorder': 'total ascending'}  # This sorts by the y-values!
)

fig_sorted.show()

> 💡 **Reflection:** Imagine you want to order the chart alphabetically in ascending order (A-Z). How would you change the code below?

In [11]:
# Step 1: Create the chart
fig_alphabetical = px.bar(
    keyword_df,
    x='keyword',
    y='score',
    title='Alphabetically Sorted Keywords',
    labels={'keyword': 'Keywords', 'score': 'Average Score'},
)

# Step 2: Sort the x-axis alphabetically
fig_alphabetical.update_layout(
    xaxis={'categoryorder': 'total descending'}
)

fig_alphabetical.show()

### Text Case
Another annoying feature of this chart is that the labels for each category are lowercase. This looks sloppy. Plotly has a way to change this. We can update the `xaxis` layout and add the key: `tickfont_textcase`. This tells us how we want the text case to appear. Our options are:

`'normal' | 'word caps' | 'upper' | 'lower'`

How would we modify the code below if we wanted to capitalize all of the words?

In [12]:
# Step 1: Create the chart
fig_textcase = px.bar(
    keyword_df,
    x='keyword',
    y='score',
    title='Keywords with Proper Text Formatting',
    labels={'keyword': 'Keywords', 'score': 'Average Score'},
)

# Step 2: Sort the x-axis by the y-values (scores)
fig_textcase.update_layout(
    xaxis={'categoryorder': 'total ascending',
           'tickfont_textcase': '???'}  # Fill in the text case option here!
)

fig_textcase.show()

ValueError: 
    Invalid value of type 'builtins.str' received for the 'textcase' property of layout.xaxis.tickfont
        Received value: '???'

    The 'textcase' property is an enumeration that may be specified as:
      - One of the following enumeration values:
            ['normal', 'word caps', 'upper', 'lower']

### Templates

It is also possible to change the entire layout of the chart in one fell swoop with templates. A template is a set of style variables that you can apply to entire chart. 

You set the template as you create the chart. The options are:

- 'ggplot2'
- 'seaborn'
- 'simple_white'
- 'plotly'
- 'plotly_white'
- 'plotly_dark'
- 'presentation'
- 'xgridoff'
- 'ygridoff'
- 'gridon'
- 'none'

Cycle through the templates below to see if there is one you like.

In [13]:
# Step 1: Create the chart with template
fig_template = px.bar(
    keyword_df,
    x='keyword',
    y='score',
    title='Keywords with Custom Template',
    labels={'keyword': 'Keywords', 'score': 'Average Score'},
    template='plotly'  # Try different templates: 'ggplot2', 'seaborn', 'plotly_dark', etc.
)

# Step 2: Sort the x-axis by the y-values (scores)
fig_template.update_layout(
    xaxis={'categoryorder': 'total ascending',
           'tickfont_textcase': 'word caps'}
)

fig_template.show()

---

## 3 Create New Visualization - Keywords Over Time

Now let's create something completely new: tracking how different keywords trend over time!

### ✍️ 3.1 Critical Activity - Building Keyword Trends Over Time

This is more challenging! We'll provide the scaffolding, but you need to fill in the missing pieces.

In [14]:
# Step 1: Choose keywords to track over time
# Feel free to modify this list!
time_keywords = ['covid', 'party', 'football', 'class']

# Step 2: Create a function to count keyword mentions by month
def count_keyword_by_month(df, keyword):
    """
    Count how many posts contain a keyword each month
    """
    # Create mask for posts containing the keyword
    has_keyword = df['text'].apply(lambda x: contains_keyword(x, keyword))
    
    # Filter DataFrame to only posts with the keyword
    keyword_posts = df[has_keyword].copy()
    
    # Group by interval and count
    monthly_keyword_counts = keyword_posts.groupby('interval', observed=True).size().reset_index(name='count')
    
    # Add keyword column for identification
    monthly_keyword_counts['keyword'] = keyword
    
    return monthly_keyword_counts

In [15]:
# Step 3: Collect data for all keywords
all_keyword_trends = []

# YOUR TURN: Complete this loop
for keyword in time_keywords:
    keyword_data = count_keyword_by_month(df, keyword)
    all_keyword_trends.append(keyword_data)

# Combine all data into one DataFrame
keyword_trends_df = pd.concat(all_keyword_trends, ignore_index=True)

# Convert interval back to datetime for plotting
keyword_trends_df['interval'] = keyword_trends_df['interval'].dt.to_timestamp()

# Check our data
keyword_trends_df.head(10)

,interval,count,keyword
0,2020-03-01,1,covid
1,2020-07-01,6,covid
2,2020-08-01,58,covid
3,2020-09-01,41,covid
4,2020-10-01,4,covid
5,2020-11-01,3,covid
6,2020-12-01,3,covid
7,2021-01-01,1,covid
8,2021-02-01,15,covid
9,2021-05-01,6,covid


In [16]:
# Step 4: Create the keywords over time visualization
# YOUR TURN: Complete this visualization

fig_trends = px.line(keyword_trends_df, 
                     x='interval', 
                     y='count', 
                     color='keyword',
                     title='Keyword Trends Over Time on r/JMU',
                     labels={'count': 'Number of Posts', 'interval': 'Time Period'},
                     markers=True,
                    template='plotly_dark')  # Try different templates here!

# Customize the layout
fig_trends.update_layout(
    xaxis_title="Time",
    yaxis_title="Posts Containing Keyword",
    legend_title="Keywords"
)

fig_trends.show()

> 💡 **Reflection:** How can we modify the function so that we show fewer keywords?

### One Color Swatch to Rule Them All

One of the practical benefits of using a visualization library like Plotly is that styling decisions can be centralized. Instead of setting colors on each chart individually, you can define a single variable and pass it everywhere.

```python
# Define your color palette once
my_colors = px.colors.qualitative.G10

# Reuse it across every chart
fig1 = px.line(df, x='interval', y='count', color='type',
               color_discrete_sequence=my_colors)

fig2 = px.bar(keyword_df, x='keyword', y='score',
              color_discrete_sequence=my_colors)
```

To switch the entire color scheme for every chart, you change **one line**. This is exactly how real-world data publications work: a news organization like The Pudding or FiveThirtyEight defines a house color palette and applies it consistently across every chart on the page. The reader recognizes the publication's visual identity instantly, and the team only has to update one variable if the brand colors change.

**Try it:** Swap `G10` for `Vivid`, `Bold`, or `Safe` in the cell above and re-run all the charts. Every chart updates at once.

---

## Lesson Summary

### Part 1: Modifying an Existing Visualization
- `fig.update_traces()` — change visual properties of chart elements (e.g., markers, line style)
- `fig.update_layout()` — change overall chart properties (e.g., title, axis labels, colors)
- `colorway` — a list of colors Plotly cycles through automatically

### Part 2: Keyword Score Chart
- `df.sort_values('col')` — sort a DataFrame before charting to control bar order
- `.str.title()` — capitalize each word in a string column
- `template=` — apply a built-in Plotly theme to change the overall chart style

### Part 3: Keywords Over Time
- Building a trend dataset by filtering and collecting results in a loop
- `pd.concat([...], ignore_index=True)` — combine a list of DataFrames into one
- `px.line(df, x=..., y=..., color=..., line_group=...)` — multi-line chart with color coding

---

➡️ **Next:** [Lesson 4.1 — Extracting Toponyms in Texts](../lesson_4_finding_locations/lesson_4_1_extracting_locations.ipynb)